<a href="https://colab.research.google.com/github/jou3487-bit/duplicados-articulos/blob/main/COMPARACION_BASE_NM_VRS_ERP_EXT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install rapidfuzz openpyxl

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd

# Oracle ya tiene encabezados correctos
df_oracle  = pd.read_excel("Base_Oracle.xlsx")
df_oracle  = df_oracle[['CODIGO', 'DESCRIPCION', 'UNIDAD']]  # descartar desc_norm si existe

# Externo NO tiene encabezados — los asignamos manualmente
df_externo = pd.read_excel("Base_Mexico.xlsx", header=None)
df_externo.columns = ['CODIGO', 'DESCRIPCION', 'UNIDAD']

# Verificar que no haya filas basura al inicio
print("=== Oracle (primeras 3 filas) ===")
print(df_oracle.head(3))
print(f"Total: {len(df_oracle):,} filas\n")

print("=== Externo (primeras 3 filas) ===")
print(df_externo.head(3))
print(f"Total: {len(df_externo):,} filas")

=== Oracle (primeras 3 filas) ===
        CODIGO                                        DESCRIPCION UNIDAD
0  10000010001  BOLSA TRANSPARENTE ESTANON ANCHO 96 cm ± 0.80 ...      U
1  10000010002  BOLSA NEGRA ANTIDESLIZANTE ANCHO 51.6 cm + 11 ...      U
2  10000010003                 BOLSA 1820 EMPAQUE GENERICO 1000 g      U
Total: 106,256 filas

=== Externo (primeras 3 filas) ===
         CODIGO                                    DESCRIPCION  UNIDAD
0        CODIGO                                    DESCRIPCION  UNIDAD
1  001000100001  TABLON DE MADERA PARA FORMALETA DE 1"X12"X3 M     PZA
2  001000100002                            TRIPLAY DE 1 GRUESO     PZA
Total: 17,729 filas


In [ ]:
print("=== Columnas reales Oracle ===")
print(df_oracle.columns.tolist())
print("\n=== Columnas reales Externo ===")
print(df_externo.columns.tolist())

=== Columnas reales Oracle ===
['CODIGO', 'DESCRIPCION', 'UNIDAD']

=== Columnas reales Externo ===
['CODIGO', 'DESCRIPCION', 'UNIDAD']


In [ ]:
import re
from rapidfuzz import fuzz, process
from fractions import Fraction

# ── Diccionario de sinónimos técnicos ──────────────────────────
SINONIMOS = {
    # Unidades de longitud
    r'\b(PULG|PULGADAS|PULGADA|IN|INCH|INCHES|\")\b': 'PULG',
    r'\b(MTS|METROS|METRO|MT|M)\b':                   'MT',
    r'\b(CMS|CENTIMETROS|CENTIMETRO|CM)\b':            'CM',
    r'\b(MMS|MILIMETROS|MILIMETRO|MM)\b':              'MM',
    r'\b(FTS|PIES|PIE|FT|FEET|FOOT|\')\b':            'FT',
    r'\b(YDS|YARDAS|YARDA|YD|YARDS)\b':               'YD',
    # Unidades de peso
    r'\b(KGS|KILOGRAMOS|KILOGRAMO|KG)\b':             'KG',
    r'\b(GRS|GRAMOS|GRAMO|GR|G)\b':                   'GR',
    r'\b(LBS|LIBRAS|LIBRA|LB)\b':                     'LB',
    r'\b(TON|TONELADAS|TONELADA|TN)\b':               'TON',
    # Unidades de volumen
    r'\b(LTS|LITROS|LITRO|LT|L)\b':                   'LT',
    r'\b(GLS|GALONES|GALON|GL|GAL)\b':                'GL',
    r'\b(MLS|MILILITROS|MILILITRO|ML)\b':              'ML',
    # Unidades de presión/electricidad
    r'\b(PSI|LB/IN2|LB/PULG2)\b':                    'PSI',
    r'\b(BAR|BARES)\b':                               'BAR',
    r'\b(AMP|AMPERES|AMPERE|AMPERIO)\b':              'AMP',
    r'\b(VOLT|VOLTIOS|VOLTIO|VT|V)\b':                'VOLT',
    r'\b(WATT|WATTS|VATIOS|W)\b':                     'WATT',
    # Materiales
    r'\b(ACERO INOX|ACERO INOXIDABLE|INOX|SS|INOXIDABLE)\b': 'INOX',
    r'\b(GALVANIZADO|GALV|HDG)\b':                    'GALV',
    r'\b(PVC|POLICLORURO)\b':                         'PVC',
    r'\b(CPVC)\b':                                    'CPVC',
    r'\b(HDPE|POLIETILENO ALTA DENSIDAD)\b':          'HDPE',
    # Conexiones
    r'\b(NPT|NPTF|TAPERED)\b':                        'NPT',
    r'\b(BSP|BSPP|BSPT)\b':                          'BSP',
    # Generales
    r'\b(NUM|NUMERO|NO\.|NRO)\b':                     'NUM',
    r'\b(DIAM|DIAMETRO|DIA)\b':                       'DIAM',
    r'\b(LONG|LONGITUD|LARGO|LG)\b':                  'LONG',
    r'\b(PRES|PRESION)\b':                            'PRES',
    r'\b(MAX|MAXIMO|MAXIMA)\b':                       'MAX',
    r'\b(MIN|MINIMO|MINIMA)\b':                       'MIN',
    r'\b(TEMP|TEMPERATURA)\b':                        'TEMP',
}

# ── Tabla de equivalencias de fracciones ───────────────────────
# Convierte cualquier forma escrita al decimal estándar con 4 decimales
# Así  1/2" = 0.5PULG = .5" = 1/2 PULG  todos quedan igual
FRACCIONES_COMUNES = {
    # Octavos
    '1/8':  '0.1250', '2/8':  '0.2500', '3/8':  '0.3750',
    '4/8':  '0.5000', '5/8':  '0.6250', '6/8':  '0.7500',
    '7/8':  '0.8750',
    # Cuartos
    '1/4':  '0.2500', '3/4':  '0.7500',
    # Medios
    '1/2':  '0.5000',
    # Dieciseisavos (tuberías hidráulicas)
    '1/16': '0.0625', '3/16': '0.1875', '5/16': '0.3125',
    '7/16': '0.4375', '9/16': '0.5625','11/16': '0.6875',
    '13/16':'0.8125','15/16': '0.9375',
    # Treintaidosavos (menos comunes pero existen)
    '1/32': '0.0313', '3/32': '0.0938',
    # Enteros con fracción: 1-1/2, 2-1/2, etc.
    '1-1/8':'1.1250','1-1/4':'1.2500','1-3/8':'1.3750',
    '1-1/2':'1.5000','1-3/4':'1.7500',
    '2-1/2':'2.5000','3-1/2':'3.5000',
    '4-1/2':'4.5000',
}

def normalizar_fracciones(texto):
    """
    Convierte fracciones a decimal estándar para comparación uniforme.
    Ejemplos:
      1/2"        → 0.5000 PULG
      3/4 PULG    → 0.7500 PULG
      1-1/2 IN    → 1.5000 PULG   (ya convertido por sinónimos)
      .5"         → 0.5000 PULG
      0.5 PULGADA → 0.5000 PULG
    """
    # Paso 1: Normalizar decimales sin cero inicial (.5 → 0.5000)
    texto = re.sub(r'(?<!\d)\.(\d+)', lambda m: f"0.{m.group(1).ljust(4,'0')}", texto)

    # Paso 2: Normalizar decimales con cero inicial (0.5 → 0.5000)
    texto = re.sub(r'\b(0\.\d+)\b', lambda m: f"{float(m.group(1)):.4f}", texto)

    # Paso 3: Reemplazar fracciones conocidas de mayor a menor longitud
    # (ordenar por longitud para evitar que "1/2" reemplace antes que "1-1/2")
    for fraccion, decimal in sorted(FRACCIONES_COMUNES.items(),
                                    key=lambda x: len(x[0]), reverse=True):
        # Escapar el guión para el regex
        patron = re.escape(fraccion)
        texto = re.sub(rf'(?<!\d){patron}(?!\d)', decimal, texto)

    # Paso 4: Convertir fracciones no listadas con cálculo dinámico
    # Captura patrones como 5/6, 7/9, etc.
    def convertir_fraccion(m):
        try:
            valor = float(Fraction(m.group(0)))
            return f"{valor:.4f}"
        except:
            return m.group(0)

    texto = re.sub(r'\b\d+/\d+\b', convertir_fraccion, texto)

    return texto

def aplicar_sinonimos(texto):
    for patron, reemplazo in SINONIMOS.items():
        texto = re.sub(patron, reemplazo, texto)
    return texto

def extraer_numeros(texto):
    """Extrae todos los valores numéricos ya normalizados"""
    return set(re.findall(r'\d+\.\d+|\d+', texto))

def extraer_numero_parte(texto):
    patrones = [
        r'\b[A-Z]{1,4}-?\d{3,}\b',
        r'\b\d{4,}\b',
        r'\b[A-Z]\d{2,}[A-Z0-9]*\b',
        r'\b\d+[A-Z]+\d*\b',
    ]
    encontrados = []
    for p in patrones:
        encontrados.extend(re.findall(p, texto))
    return set(encontrados)

def normalizar(texto):
    texto = str(texto).upper().strip()
    texto = re.sub(r'\s+', ' ', texto)
    # Mantener /, ., - para fracciones antes de procesar
    texto = re.sub(r'[^\w\s/\.\-]', ' ', texto)
    texto = aplicar_sinonimos(texto)      # primero sinónimos (convierte " → PULG)
    texto = normalizar_fracciones(texto)  # luego fracciones (1/2 PULG → 0.5000 PULG)
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

def primera_palabra(texto):
    stopwords = {'DE', 'LA', 'EL', 'EN', 'Y', 'A', 'CON', 'PARA', 'POR', 'DEL', 'LOS', 'LAS'}
    for p in texto.split():
        if p not in stopwords and len(p) >= 3:
            return p
    return texto.split()[0] if texto.split() else 'OTROS'

# ── Aplicar a ambos dataframes ──────────────────────────────────
df_oracle['desc_norm']  = df_oracle['DESCRIPCION'].apply(normalizar)
df_externo['desc_norm'] = df_externo['DESCRIPCION'].apply(normalizar)
df_oracle['bloque']     = df_oracle['desc_norm'].apply(primera_palabra)
df_externo['bloque']    = df_externo['desc_norm'].apply(primera_palabra)
df_oracle['numeros']    = df_oracle['desc_norm'].apply(extraer_numeros)
df_externo['numeros']   = df_externo['desc_norm'].apply(extraer_numeros)
df_oracle['nro_parte']  = df_oracle['desc_norm'].apply(extraer_numero_parte)
df_externo['nro_parte'] = df_externo['desc_norm'].apply(extraer_numero_parte)

# ── Verificación visual ─────────────────────────────────────────
print("✅ Normalización completa\n")
print("Ejemplos de conversión de fracciones:")
ejemplos = [
    'VALVULA BOLA 1/2" NPT',
    'TUBERIA PVC 3/4 PULGADA',
    'CODO 1-1/2 IN 90 GRADOS',
    'NIPLE ACERO 2" X 6"',
    'REDUCCION 3/4 A 1/2 PULG',
    'VALVULA COMPUERTA .5 PULG',
]
for e in ejemplos:
    print(f"  ANTES:  {e}")
    print(f"  DESPUÉS:{normalizar(e)}\n")

print(f"Bloques Oracle:  {df_oracle['bloque'].nunique():,} grupos")
print(f"Bloques Externo: {df_externo['bloque'].nunique():,} grupos")

✅ Normalización completa

Ejemplos de conversión de fracciones:
  ANTES:  VALVULA BOLA 1/2" NPT
  DESPUÉS:VALVULA BOLA 0.5000 NPT

  ANTES:  TUBERIA PVC 3/4 PULGADA
  DESPUÉS:TUBERIA PVC 0.7500 PULG

  ANTES:  CODO 1-1/2 IN 90 GRADOS
  DESPUÉS:CODO 1.5000 PULG 90 GRADOS

  ANTES:  NIPLE ACERO 2" X 6"
  DESPUÉS:NIPLE ACERO 2 X 6

  ANTES:  REDUCCION 3/4 A 1/2 PULG
  DESPUÉS:REDUCCION 0.7500 A 0.5000 PULG

  ANTES:  VALVULA COMPUERTA .5 PULG
  DESPUÉS:VALVULA COMPUERTA 0.5000 PULG

Bloques Oracle:  9,580 grupos
Bloques Externo: 3,028 grupos


In [ ]:
UMBRAL = 85  # Ligeramente más bajo porque el scoring ajustado compensa

def calcular_score_final(score_base, row_ext, row_ora):
    """
    Ajusta el score base con bonificaciones y penalizaciones
    según números, unidades y número de parte
    """
    bonus = 0
    detalle = []

    # ── Unidad de medida ───────────────────────────────────────
    unidad_ext = str(row_ext['UNIDAD']).upper().strip()
    unidad_ora = str(row_ora['UNIDAD']).upper().strip()
    if unidad_ext == unidad_ora:
        bonus += 3
        detalle.append("✅ Unidad igual")
    else:
        bonus -= 5  # Penalización: misma desc pero unidades distintas es sospechoso
        detalle.append(f"⚠️ Unidad distinta ({unidad_ext} vs {unidad_ora})")

    # ── Números y dimensiones ──────────────────────────────────
    nums_ext = row_ext['numeros']
    nums_ora = row_ora['numeros']

    if nums_ext and nums_ora:
        if nums_ext == nums_ora:
            bonus += 5  # Todos los números coinciden exactamente
            detalle.append("✅ Dimensiones exactas")
        elif nums_ext & nums_ora:  # intersección
            coincidencia_nums = len(nums_ext & nums_ora) / max(len(nums_ext), len(nums_ora))
            bonus += round(coincidencia_nums * 4)
            detalle.append(f"⚠️ Dimensiones parciales ({len(nums_ext & nums_ora)}/{max(len(nums_ext), len(nums_ora))})")
        else:
            bonus -= 8  # Números completamente distintos = probablemente NO es el mismo artículo
            detalle.append("❌ Dimensiones distintas")
    elif nums_ext and not nums_ora:
        bonus -= 3
        detalle.append("⚠️ Externo tiene dims, Oracle no")
    elif not nums_ext and nums_ora:
        bonus -= 3
        detalle.append("⚠️ Oracle tiene dims, Externo no")

    # ── Número de parte ────────────────────────────────────────
    np_ext = row_ext['nro_parte']
    np_ora = row_ora['nro_parte']

    if np_ext and np_ora:
        if np_ext & np_ora:
            bonus += 8  # Número de parte coincide = muy fuerte señal
            detalle.append("✅ Núm. de parte coincide")
        else:
            bonus -= 10  # Números de parte distintos = artículos distintos
            detalle.append("❌ Núm. de parte distinto")

    score_final = min(100, max(0, score_base + bonus))
    return score_final, " | ".join(detalle)


# ── Loop principal ─────────────────────────────────────────────
resultados = []
bloques = df_externo['bloque'].unique()
total = len(bloques)

for i, bloque in enumerate(bloques):
    if i % 50 == 0:
        print(f"  Progreso: {i+1}/{total} grupos | Pares: {len(resultados):,}")

    sub_oracle  = df_oracle[df_oracle['bloque'] == bloque].reset_index(drop=True)
    sub_externo = df_externo[df_externo['bloque'] == bloque]

    if sub_oracle.empty:
        bloques_oracle = df_oracle['bloque'].unique()
        matches_b = process.extract(bloque, bloques_oracle,
                                    scorer=fuzz.ratio, limit=2, score_cutoff=80)
        if not matches_b:
            continue
        sub_oracle = pd.concat(
            [df_oracle[df_oracle['bloque'] == m[0]] for m in matches_b]
        ).reset_index(drop=True)

    oracle_descs = sub_oracle['desc_norm'].tolist()

    for _, row_ext in sub_externo.iterrows():
        matches = process.extract(
            row_ext['desc_norm'],
            oracle_descs,
            scorer=fuzz.token_sort_ratio,
            limit=3,
            score_cutoff=UMBRAL
        )
        for _, score_base, idx in matches:
            row_ora = sub_oracle.iloc[idx]
            score_final, detalle = calcular_score_final(score_base, row_ext, row_ora)

            # Filtrar resultados que bajaron mucho por penalizaciones
            if score_final < 80:
                continue

            resultados.append({
                "CODIGO_EXTERNO":  row_ext['CODIGO'],
                "DESC_EXTERNO":    row_ext['DESCRIPCION'],
                "UNIDAD_EXTERNO":  row_ext['UNIDAD'],
                "CODIGO_ORACLE":   row_ora['CODIGO'],
                "DESC_ORACLE":     row_ora['DESCRIPCION'],
                "UNIDAD_ORACLE":   row_ora['UNIDAD'],
                "SCORE_BASE":      score_base,
                "SCORE_FINAL":     score_final,
                "DETALLE":         detalle,
                "DECISION":        "🔴 DUPLICADO"    if score_final >= 95
                                   else "⚠️ REVISAR"  if score_final >= 85
                                   else "🟡 POSIBLE"
            })

df_resultado = pd.DataFrame(resultados).sort_values('SCORE_FINAL', ascending=False)

print(f"\n✅ Comparación completa")
print(f"   🔴 Probables duplicados (≥95%): {len(df_resultado[df_resultado['SCORE_FINAL'] >= 95]):,}")
print(f"   ⚠️  Para revisar (85-94%):      {len(df_resultado[(df_resultado['SCORE_FINAL'] >= 85) & (df_resultado['SCORE_FINAL'] < 95)]):,}")
print(f"   🟡 Posibles (80-84%):           {len(df_resultado[df_resultado['SCORE_FINAL'] < 85]):,}")
print(f"   📊 Total pares encontrados:     {len(df_resultado):,}")

  Progreso: 1/3028 grupos | Pares: 0
  Progreso: 51/3028 grupos | Pares: 253
  Progreso: 101/3028 grupos | Pares: 422
  Progreso: 151/3028 grupos | Pares: 447
  Progreso: 201/3028 grupos | Pares: 486
  Progreso: 251/3028 grupos | Pares: 540
  Progreso: 301/3028 grupos | Pares: 576
  Progreso: 351/3028 grupos | Pares: 615
  Progreso: 401/3028 grupos | Pares: 623
  Progreso: 451/3028 grupos | Pares: 650
  Progreso: 501/3028 grupos | Pares: 773
  Progreso: 551/3028 grupos | Pares: 804
  Progreso: 601/3028 grupos | Pares: 810
  Progreso: 651/3028 grupos | Pares: 816
  Progreso: 701/3028 grupos | Pares: 826
  Progreso: 751/3028 grupos | Pares: 832
  Progreso: 801/3028 grupos | Pares: 837
  Progreso: 851/3028 grupos | Pares: 839
  Progreso: 901/3028 grupos | Pares: 842
  Progreso: 951/3028 grupos | Pares: 843
  Progreso: 1001/3028 grupos | Pares: 852
  Progreso: 1051/3028 grupos | Pares: 855
  Progreso: 1101/3028 grupos | Pares: 862
  Progreso: 1151/3028 grupos | Pares: 887
  Progreso: 1201/

In [ ]:
from google.colab import files

with pd.ExcelWriter("posibles_duplicados.xlsx", engine='openpyxl') as writer:
    df_resultado.to_excel(writer, index=False, sheet_name='Duplicados')

    resumen = pd.DataFrame({
        'Métrica': [
            'Total artículos Oracle',
            'Total artículos Externo',
            '🔴 Probables duplicados (≥95%)',
            '⚠️  Para revisar (85-94%)',
            '🟡 Posibles (80-84%)',
            '✅ Sin coincidencia en Oracle'
        ],
        'Cantidad': [
            len(df_oracle),
            len(df_externo),
            len(df_resultado[df_resultado['SCORE_FINAL'] >= 95]),
            len(df_resultado[(df_resultado['SCORE_FINAL'] >= 85) &
                             (df_resultado['SCORE_FINAL'] < 95)]),
            len(df_resultado[df_resultado['SCORE_FINAL'] < 85]),
            len(df_externo) - df_resultado['CODIGO_EXTERNO'].nunique()
        ]
    })
    resumen.to_excel(writer, index=False, sheet_name='Resumen')

files.download("posibles_duplicados.xlsx")
print("✅ Descarga iniciada")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Descarga iniciada
